|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 4. The scheduler decides who runs, who waits, who is
preempted, and how many tokens go into each step. Its failures do not give
wrong text. They give time: stalls, storms, starvation and latency that the
dashboard does not show.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 11. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 4.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| Machine | Number |
|---|---|
| Your card, prefill | about 37 TFLOP/s sustained (Part 1) |
| PCIe 4.0 x16, pinned memory | about 25 GB/s |

| Model | Parameters | KV bytes per token | bf16 weights |
|---|---|---|---|
| Qwen3-1.7B | 1.72 B | 114,688 (112 KiB) | 3.44 GB |

- The block size is 16 tokens.
- A prefill of N tokens costs about `2 x parameters x N` FLOP, plus the
  attention.
- A decode step at a small batch costs about 25 ms on your card with this
  model.

# Ticket 1: the preemption storm

**Severity:** high. **Reported by:** the SRE team.

> At the peak, 38% of the requests are preempted at least once, and the
> throughput falls by half. We raised `max_num_seqs` from 128 to 256 to
> admit more requests, and it got worse.

**Evidence**

- The pool has 12,000 blocks.
- The admission rule: admit a request when the free blocks cover its
  **prompt**.
- The peak workload: prompts of about 1,000 tokens and answers of about
  1,500 tokens.
- At the start of the peak, the scheduler admitted 190 requests at once.
- The preemption mode is recompute.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: the freeze after a preemption

**Severity:** medium. **Reported by:** users of the chat service.

> At the peak, all of our streams freeze for about two seconds, a few
> times each hour.

**Evidence**

- Qwen3-1.7B. The service accepts long documents of up to 12,000 tokens.
  [Chunked prefill](../../GLOSSARY.md#chunked-prefill) is on for new requests, with a chunk of 512 tokens.
- Each freeze starts at the same step as a line in the log:

      resumed seq 88213 (recompute, 12000 tokens)

- The code that resumes a preempted request:

  ```python
  def resume(self, seq):
      tokens = seq.prompt + seq.output
      self.prefill(seq, tokens)          # one forward pass
  ```

- One engineer suspects Python garbage collection pauses.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: the good p99 and the angry users

**Severity:** medium. **Reported by:** the product team.

> Users say that the text freezes for two seconds, a few times during a
> long answer. The dashboard says that the p99 of the time between
> tokens is 24 ms. The users must be wrong.

**Evidence**

- The server runs 64 streams. The time between tokens: p50 22 ms, p99
  24 ms.
- A document of 16,000 tokens arrives about every 3 minutes. Chunked
  prefill is **off**.
- The time to the first token of the documents is 2.3 s. The document
  users are happy.
- The p99.9 is 25 ms.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: chunked prefill made the documents slow

**Severity:** medium. **Reported by:** the team of the document
service.

> Since chunked prefill, the time to the first token of our documents of
> 16,000 tokens went from 2.3 s to 6.4 s. The chat team is happy, and we
> are not.

**Evidence**

- The token budget for each step is 128 tokens.
- About 64 decode streams run all the time. Each decode takes one token
  of the budget.
- A step with the budget of 128 takes about 25 ms.
- The time between tokens for the chat streams is now 25 ms at the p99.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: the request that never finishes

**Severity:** medium. **Reported by:** a customer.

> One of my requests took 41 minutes. The others take 30 seconds.

**Evidence**

- The request had a prompt of 2,000 tokens. The log shows it was
  preempted 37 times.
- The scheduler admits a request when the free blocks cover its prompt.
  At the peak it preempts about 8 times each second.
- The scheduler preempts the request that it admitted **last**.
- The code that preempts:

  ```python
  def preempt(self, seq):
      self.free(seq)
      self.running.remove(seq)
      self.waiting.append(seq)          # to the back of the queue
  ```

- The engineer proposes a fix: put the preempted request at the front
  of the queue. The engineer also thinks that long prompts are unlucky,
  because they take more blocks.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: long prompts forget their beginning

**Severity:** high. **Reported by:** the evaluation team.

> Since chunked prefill, the answers to long prompts are fluent but
> wrong. They ignore the instructions at the start of the prompt. Short
> prompts are fine.

**Evidence**

- The chunk size is 512 tokens.
- The fraction of wrong answers by prompt length:

  | prompt tokens | wrong |
  |---|---|
  | up to 512 | 0% |
  | 513 to 1,024 | 61% |
  | above 1,024 | 88% |

- The code that prepares a chunk:

  ```python
  positions = torch.arange(len(chunk_tokens))
  ```

- The team says that long prompts are harder questions.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: chunked prefill costs 8% of the throughput

**Severity:** low. **Reported by:** the finance team.

> After chunked prefill, the tokens/s fell by 8%. That is 8% more GPUs
> for the same traffic. Turn it off.

**Evidence**

| | before | after |
|---|---|---|
| tokens/s | 9,400 | 8,650 |
| time between tokens, p99 | 610 ms | 95 ms |
| requests that meet the promise (TTFT < 3 s and TPOT < 100 ms) | 71% | 96% |

- The promise to the customers is in the last row.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 8: the swap that runs at a quarter of the bus

**Severity:** low. **Reported by:** the kernel team.

> We use swap for long sequences, as Ticket 2 suggests. A swap-out of
> 12,000 tokens takes 230 ms. We expected about 55 ms.

**Evidence**

- The CPU copy of the cache:

  ```python
  cpu_blocks = torch.empty(num_cpu_blocks, 16, kv_heads, head_dim, dtype=torch.bfloat16)
  ...
  cpu_blocks[dst].copy_(gpu_blocks[src], non_blocking=True)
  ```

- Nsight Systems names the copy `Memcpy DtoH (Pageable)`.
- The bus is PCIe 4.0 x16. The host has 512 GB of RAM, and 380 GB are
  free.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**